# critic_model.py Validation

**Intent:** Gut-check the per-critic KDE lambda model (`critic_model.py`) against resolved movies.
Verify that profiles, KDEs, lambda estimates, and p_fresh estimates are sensible before using in production.

**Plan reference:** `plans/plan_critic_kde_lambda.md` Step 6.

**Validation movies** (minute-level timestamp data):
they_will_kill_you, forbidden_fruits_2026, project_hail_mary, ready_or_not_2_here_i_come

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta

from rotten_tomatoes_forecasting import (
    build_critic_profiles, build_kde_lambda_model,
    estimate_lambda, estimate_p_fresh, default_training_slugs,
)

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})

ROOT = Path('..').resolve()

# -- Load data -----------------------------------------------------------------
reviews = pd.read_csv(ROOT / 'reviews.csv')
reviews['estimated_timestamp'] = pd.to_datetime(
    reviews['estimated_timestamp'], format='ISO8601', utc=True
)
movies = pd.read_csv(ROOT / 'movies_index.csv')
movies['Bet Close Date'] = pd.to_datetime(movies['Bet Close Date'], utc=True)
movies['Bet Open Date'] = pd.to_datetime(movies['Bet Open Date'], utc=True)

print(f"Reviews: {len(reviews):,}")
print(f"Movies: {len(movies)}")

## 1. Build profiles and KDE model

Using 20 most recent resolved movies as training set. Print sanity checks from the plan:
- `sum(base_rate)` should approximate mean total reviews per movie
- All fresh rates in [0, 1]
- KDE fit counts (empirical vs fallback) should be similar to notebook prototype (392 / 46 / 297)

In [ ]:
# Build with a generic training set (no specific target movie excluded)
training_slugs = default_training_slugs(movies, n=20)
print(f"Training slugs ({len(training_slugs)}):")
for s in training_slugs:
    print(f"  {s}")

profiles = build_critic_profiles(reviews, movies, training_slugs)
model = build_kde_lambda_model(profiles)

# Additional sanity checks
df = profiles.df
print(f"\nAll fresh_rate in [0,1]: {df['fresh_rate'].between(0, 1).all()}")
print(f"Fresh rate distribution: {df['fresh_rate'].describe().round(3).to_dict()}")
print(f"Timing data range: {min(min(t) for t in df['timing_data']):.1f} - {max(max(t) for t in df['timing_data']):.1f} days before close")

## 2. Population prior shape

Should peak 3-7 days before close (consistent with critics_index.ipynb findings).

In [ ]:
t_grid = np.linspace(0, 30, 500)
pop_density = model.population_prior(t_grid)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(t_grid, pop_density, alpha=0.3, color='steelblue')
ax.plot(t_grid, pop_density, 'steelblue', lw=2)
ax.set_xlabel('Days before close')
ax.set_ylabel('Density')
ax.set_title('Population prior KDE (all critics pooled)')
ax.invert_xaxis()
ax.axvline(t_grid[np.argmax(pop_density)], color='red', ls='--', alpha=0.5,
           label=f'Peak: {t_grid[np.argmax(pop_density)]:.1f}d')
ax.legend()
plt.tight_layout()

## 3. Lambda curve (no observed reviews)

Expected remaining reviews as a function of days-before-close, with no reviews observed yet.
Compare to prototype from critics_index.ipynb: T-7d=83.7, T-3d=24.3, T-1d=2.4.

In [ ]:
# Compute expected remaining reviews curve (no observations)
dbc_grid = np.linspace(0.5, 30, 200)
expected_remaining = []
for dbc in dbc_grid:
    htc = dbc * 24
    lam = estimate_lambda(model, dbc, htc, observed_critics=set())
    expected_remaining.append(lam * htc)

expected_remaining = np.array(expected_remaining)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dbc_grid, expected_remaining, 'steelblue', lw=2)
ax.set_xlabel('Days before close')
ax.set_ylabel('Expected remaining reviews')
ax.set_title('Aggregate lambda curve (KDE model, no reviews observed)')
ax.invert_xaxis()

# Mark reference points
for d, ref in [(7, 83.7), (3, 24.3), (1, 2.4)]:
    idx = np.argmin(np.abs(dbc_grid - d))
    val = expected_remaining[idx]
    ax.plot(d, val, 'ro', ms=8)
    ax.annotate(f'T-{d}d: {val:.1f} (proto: {ref})', (d, val),
                textcoords='offset points', xytext=(10, 10), fontsize=9)

ax.grid(True, alpha=0.3)
plt.tight_layout()

# Print values at key horizons
for d in [7, 3, 1, 0.5]:
    idx = np.argmin(np.abs(dbc_grid - d))
    htc = d * 24
    lam = estimate_lambda(model, d, htc, observed_critics=set())
    print(f"T-{d}d: lambda={lam:.3f} rev/hr, expected_remaining={lam*htc:.1f}")

## 4. Historical validation: predicted vs actual remaining reviews

For resolved movies, simulate the model at T-7d, T-3d, T-1d snapshots.
At each snapshot, compute:
- Which critics have reviewed (observed_critics)
- Expected remaining reviews from the model
- Actual remaining reviews (ground truth)

Prioritize the 6 movies with minute-level timestamps, but include all resolved movies for broader coverage.

In [ ]:
# Simulate model at historical snapshots for resolved movies
close_map = movies.set_index('Slug')['Bet Close Date'].to_dict()
snapshots_dbc = [7, 3, 1]  # days before close

# Use all resolved movies that aren't in the training set for out-of-sample validation
now = pd.Timestamp.now(tz='UTC')
resolved = movies[movies['Bet Close Date'] < now].copy()
# For each test movie, rebuild model excluding it (leave-one-out)
# For speed, we'll use the generic model and note that the test movie IS in training
# A proper LOO would rebuild per movie — flag this as a simplification

results = []

for _, movie in resolved.iterrows():
    slug = movie['Slug']
    bet_close = movie['Bet Close Date']

    # All reviews for this movie, before bet close
    mr = reviews[
        (reviews['movie_slug'] == slug) &
        (reviews['estimated_timestamp'] < bet_close)
    ].copy()
    if len(mr) < 5:
        continue

    mr['dbc'] = (bet_close - mr['estimated_timestamp']).dt.total_seconds() / 86400
    total_final = len(mr)
    first_review_dbc = mr['dbc'].max()

    for snap_dbc in snapshots_dbc:
        # Reviews seen at this snapshot
        seen = mr[mr['dbc'] > snap_dbc]
        remaining_actual = len(mr[mr['dbc'] <= snap_dbc])
        observed_critics = set(seen['reviewer_name'].unique())
        observed_count = len(seen)

        if observed_count == 0:
            continue

        htc = snap_dbc * 24
        lam = estimate_lambda(
            model, snap_dbc, htc, observed_critics,
            observed_count=observed_count, first_review_dbc=first_review_dbc,
        )
        expected_rem = lam * htc

        # Also compute unscaled for comparison
        lam_unscaled = estimate_lambda(
            model, snap_dbc, htc, observed_critics,
        )
        expected_rem_unscaled = lam_unscaled * htc

        results.append({
            'slug': slug,
            'snap_dbc': snap_dbc,
            'total_final': total_final,
            'observed': observed_count,
            'remaining_actual': remaining_actual,
            'remaining_predicted_scaled': expected_rem,
            'remaining_predicted_unscaled': expected_rem_unscaled,
        })

rdf = pd.DataFrame(results)
print(f"Snapshots computed: {len(rdf)} across {rdf['slug'].nunique()} movies")
rdf.head(10)

In [ ]:
# Predicted vs actual remaining reviews — scatter plots per snapshot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, snap in zip(axes, snapshots_dbc):
    sub = rdf[rdf['snap_dbc'] == snap]
    lim = max(sub['remaining_actual'].max(), sub['remaining_predicted_scaled'].max()) * 1.1

    ax.scatter(sub['remaining_actual'], sub['remaining_predicted_scaled'],
               alpha=0.5, s=30, label='Scaled', color='steelblue')
    ax.scatter(sub['remaining_actual'], sub['remaining_predicted_unscaled'],
               alpha=0.3, s=20, label='Unscaled', color='orange', marker='x')
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='Perfect')
    ax.set_xlabel('Actual remaining')
    ax.set_ylabel('Predicted remaining')
    ax.set_title(f'T-{snap}d')
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_aspect('equal')

plt.suptitle('Predicted vs actual remaining reviews at each snapshot', fontsize=14)
plt.tight_layout()

# Print error stats
for snap in snapshots_dbc:
    sub = rdf[rdf['snap_dbc'] == snap]
    err_s = sub['remaining_predicted_scaled'] - sub['remaining_actual']
    err_u = sub['remaining_predicted_unscaled'] - sub['remaining_actual']
    print(f"\nT-{snap}d (n={len(sub)}):")
    print(f"  Scaled   — MAE={err_s.abs().mean():.1f}, median err={err_s.median():.1f}, RMSE={np.sqrt((err_s**2).mean()):.1f}")
    print(f"  Unscaled — MAE={err_u.abs().mean():.1f}, median err={err_u.median():.1f}, RMSE={np.sqrt((err_u**2).mean()):.1f}")

## 5. Scaling factor distribution

For each snapshot, what's the observed/expected scaling factor? Should center near 1.0 with reasonable spread.

In [ ]:
# Compute scaling factors for each movie at each snapshot
from rotten_tomatoes_forecasting.critic_model import _compute_scaling, _blended_integral

scaling_results = []

for _, movie in resolved.iterrows():
    slug = movie['Slug']
    bet_close = movie['Bet Close Date']

    mr = reviews[
        (reviews['movie_slug'] == slug) &
        (reviews['estimated_timestamp'] < bet_close)
    ].copy()
    if len(mr) < 5:
        continue

    mr['dbc'] = (bet_close - mr['estimated_timestamp']).dt.total_seconds() / 86400
    first_review_dbc = mr['dbc'].max()

    for snap_dbc in snapshots_dbc:
        seen = mr[mr['dbc'] > snap_dbc]
        observed_count = len(seen)
        if observed_count == 0:
            continue

        scaling = _compute_scaling(model, snap_dbc, observed_count, first_review_dbc)
        scaling_results.append({
            'slug': slug, 'snap_dbc': snap_dbc,
            'scaling': scaling, 'observed': observed_count,
        })

sdf = pd.DataFrame(scaling_results)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, snap in zip(axes, snapshots_dbc):
    sub = sdf[sdf['snap_dbc'] == snap]
    ax.hist(sub['scaling'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(1.0, color='red', ls='--', lw=2, label='1.0')
    ax.axvline(sub['scaling'].median(), color='orange', ls='-', lw=2,
               label=f'Median: {sub["scaling"].median():.2f}')
    ax.set_xlabel('Scaling factor')
    ax.set_ylabel('Count')
    ax.set_title(f'T-{snap}d (n={len(sub)})')
    ax.legend(fontsize=9)

plt.suptitle('Observed/expected scaling factor distribution', fontsize=14)
plt.tight_layout()

## 6. p_fresh validation

At T-1d for resolved movies, p_fresh should approximate the final tomatometer score / 100.
Prior p_fresh (no reviews observed) should be in 0.6-0.8 range.

In [ ]:
# Prior p_fresh (no observations)
prior_pf = estimate_p_fresh(profiles, set(), 0, 0)
print(f"Prior p_fresh (no obs): {prior_pf:.4f}  (expect 0.6-0.8)")

# p_fresh at T-1d for resolved movies vs final score
pf_results = []
for _, movie in resolved.iterrows():
    slug = movie['Slug']
    bet_close = movie['Bet Close Date']

    mr = reviews[
        (reviews['movie_slug'] == slug) &
        (reviews['estimated_timestamp'] < bet_close)
    ].copy()
    if len(mr) < 10:
        continue

    mr['dbc'] = (bet_close - mr['estimated_timestamp']).dt.total_seconds() / 86400
    total_final = len(mr)
    fresh_final = (mr['tomatometer_sentiment'] == 'positive').sum()
    final_score = fresh_final / total_final

    # Snapshot at T-1d
    snap_dbc = 1.0
    seen = mr[mr['dbc'] > snap_dbc]
    fresh_seen = (seen['tomatometer_sentiment'] == 'positive').sum()
    total_seen = len(seen)
    observed_critics = set(seen['reviewer_name'].unique())

    if total_seen == 0:
        continue

    pf = estimate_p_fresh(profiles, observed_critics, fresh_seen, total_seen)
    pf_results.append({
        'slug': slug, 'p_fresh_t1d': pf,
        'final_score': final_score, 'total_final': total_final,
    })

pfdf = pd.DataFrame(pf_results)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(pfdf['final_score'], pfdf['p_fresh_t1d'], alpha=0.5, s=30, color='steelblue')
lim = (0.2, 1.05)
ax.plot(lim, lim, 'k--', alpha=0.3, label='Perfect')
ax.set_xlabel('Final tomatometer score (fresh/total)')
ax.set_ylabel('p_fresh estimate at T-1d')
ax.set_title(f'p_fresh at T-1d vs final score (n={len(pfdf)})')
ax.set_xlim(*lim)
ax.set_ylim(*lim)
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()

err = pfdf['p_fresh_t1d'] - pfdf['final_score']
print(f"\np_fresh at T-1d vs final score:")
print(f"  MAE={err.abs().mean():.4f}, median err={err.median():.4f}")
print(f"  Correlation: {pfdf['p_fresh_t1d'].corr(pfdf['final_score']):.3f}")

## 7. Deep dive: minute-level timestamp movies

Detailed view for the validation movies with the best timestamp data.

In [ ]:
# Detailed lambda + p_fresh trajectories for priority validation movies
priority_slugs = [
    'they_will_kill_you', 'forbidden_fruits_2026',
    'project_hail_mary', 'ready_or_not_2_here_i_come',
]
# Filter to ones that are resolved and in our data
priority_slugs = [s for s in priority_slugs if s in close_map and close_map[s] < now]

fig, axes = plt.subplots(len(priority_slugs), 2, figsize=(16, 5 * len(priority_slugs)))
if len(priority_slugs) == 1:
    axes = axes.reshape(1, -1)

for i, slug in enumerate(priority_slugs):
    bet_close = close_map[slug]
    mr = reviews[
        (reviews['movie_slug'] == slug) &
        (reviews['estimated_timestamp'] < bet_close)
    ].copy()
    mr['dbc'] = (bet_close - mr['estimated_timestamp']).dt.total_seconds() / 86400
    total_final = len(mr)
    fresh_final = (mr['tomatometer_sentiment'] == 'positive').sum()
    first_review_dbc = mr['dbc'].max()

    # Trajectory: evaluate at many time points
    dbc_traj = np.linspace(0.5, min(first_review_dbc - 0.5, 20), 50)
    lambdas_s, lambdas_u, pfreshes, actual_rem = [], [], [], []

    for dbc in dbc_traj:
        htc = dbc * 24
        seen = mr[mr['dbc'] > dbc]
        observed_critics = set(seen['reviewer_name'].unique())
        fresh_seen = (seen['tomatometer_sentiment'] == 'positive').sum()
        total_seen = len(seen)
        remaining = len(mr[mr['dbc'] <= dbc])

        lam_s = estimate_lambda(model, dbc, htc, observed_critics,
                                observed_count=total_seen, first_review_dbc=first_review_dbc)
        lam_u = estimate_lambda(model, dbc, htc, observed_critics)
        pf = estimate_p_fresh(profiles, observed_critics, fresh_seen, total_seen)

        lambdas_s.append(lam_s * htc)
        lambdas_u.append(lam_u * htc)
        pfreshes.append(pf)
        actual_rem.append(remaining)

    # Left: lambda trajectory
    ax = axes[i, 0]
    ax.plot(dbc_traj, actual_rem, 'k-', lw=2, label='Actual remaining')
    ax.plot(dbc_traj, lambdas_s, 'steelblue', lw=2, label='Predicted (scaled)')
    ax.plot(dbc_traj, lambdas_u, 'orange', lw=1.5, ls='--', label='Predicted (unscaled)')
    ax.set_xlabel('Days before close')
    ax.set_ylabel('Remaining reviews')
    ax.set_title(f'{slug} — lambda trajectory ({total_final} total reviews)')
    ax.invert_xaxis()
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # Right: p_fresh trajectory
    ax = axes[i, 1]
    ax.plot(dbc_traj, pfreshes, 'steelblue', lw=2, label='p_fresh estimate')
    ax.axhline(fresh_final / total_final, color='red', ls='--', lw=1.5,
               label=f'Final score: {fresh_final/total_final:.3f}')
    ax.set_xlabel('Days before close')
    ax.set_ylabel('p_fresh')
    ax.set_title(f'{slug} — p_fresh trajectory')
    ax.invert_xaxis()
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(0, 1)

plt.tight_layout()